# 06 — Capstone: end-to-end lakehouse pipeline (EMR)

Run the full batch + streaming medallion pipeline as a single flow and validate outputs. This mirrors what the MWAA DAG (`airflow/dags/retail_lakehouse_pipeline.py`) runs as a scheduled pipeline in production.

Run the cell below first, before anything else -- it configures Delta Lake for this notebook's Spark session (EMR doesn't bundle Delta by default, unlike Databricks). `%%configure -f` must run before any other Spark code in this session.

In [ ]:
%%configure -f
{"conf": {"spark.jars.packages": "io.delta:delta-spark_2.12:3.1.0", "spark.sql.extensions": "io.delta.sql.DeltaSparkSessionExtension", "spark.sql.catalog.spark_catalog": "org.apache.spark.sql.delta.catalog.DeltaCatalog"}}

In [ ]:
schema = "retail_lakehouse"
base_path = "s3://<your-lakehouse-bucket>/data"

from retail_lakehouse.config import PipelineConfig
cfg = PipelineConfig(schema=schema, base_path=base_path)
spark.sql(f"USE `{cfg.schema}`")

## Seed + batch bronze/silver/gold

In [ ]:
from retail_lakehouse.generate import synthetic_products, synthetic_customers, synthetic_events
from retail_lakehouse.transformations import (
    normalize_click_events, filter_valid_events, deduplicate_events,
    enrich_with_product_customer, revenue_by_hour, add_ingest_metadata,
)
from pyspark.sql import functions as F
import random
random.seed(11)

products = spark.createDataFrame(synthetic_products(50))
customers = spark.createDataFrame(synthetic_customers(200))
events = spark.createDataFrame(list(synthetic_events(5000)))

products.write.mode("overwrite").format("delta").saveAsTable(cfg.table("dim_product_seed"))
customers.write.mode("overwrite").format("delta").saveAsTable(cfg.table("dim_customer_seed"))

bronze = add_ingest_metadata(events, "capstone_batch")
silver = deduplicate_events(filter_valid_events(normalize_click_events(bronze)))
silver.write.mode("overwrite").format("delta").partitionBy("event_date").saveAsTable(cfg.table("capstone_silver_events"))

enriched = enrich_with_product_customer(silver, products, customers)
gold = revenue_by_hour(enriched.withColumn("is_purchase", F.col("event_type").isin("purchase", "checkout")))
gold.write.mode("overwrite").format("delta").saveAsTable(cfg.table("capstone_gold_revenue"))

## Validate outputs

A real production run would fail the pipeline (non-zero exit) on a failed assertion -- exactly what `emr_jobs/06_capstone_end_to_end.py` does, which is what the MWAA DAG actually schedules. This notebook version just raises, which is the right behavior interactively too.

In [ ]:
from retail_lakehouse.quality import assert_no_duplicate_keys

assert spark.table(cfg.table("capstone_silver_events")).count() > 0, "silver produced no rows"
assert spark.table(cfg.table("capstone_gold_revenue")).count() > 0, "gold produced no rows"
assert_no_duplicate_keys(spark.table(cfg.table("capstone_silver_events")), ["event_id"])

print(spark.table(cfg.table("capstone_gold_revenue")).orderBy(F.desc("revenue")).toPandas())
print("Capstone pipeline validated successfully.")

## Discussion questions

1. Which parts of this pipeline should move from `availableNow` scheduled streaming to a true always-on continuous stream, and what would that cost in cluster spend vs latency improvement?
2. Where should MSK credentials live, and how would you rotate them without downtime?
3. Which tables should be governed as Delta vs Iceberg if a second (non-EMR) query engine joins this platform?
4. How would you test a schema change to the clickstream event contract without breaking consumers?
5. What SLOs would you define for streaming lag and gold table freshness, and how would you alert on them?
6. If MSK ingestion falls behind for an hour then catches up, what do you need to verify about the gold table afterward?